# BladesTime — Blades of Time (Nintendo Switch, NSP base + update → iOS 26)

Fluxo **idêntico ao notebook original Dante2iOS_DMC2_LinuxNativeAOT**, adaptado para Blades of Time: **Colab → Google Drive → Colab → GitHub Actions → Google Drive**. Mantidas as **13 células originais**.

**Fonte:** subpasta **`botime`** da pasta compartilhada já cadastrada. Existem **2 NSPs**: base `Blades of Time [0100CFA00CC74000][v0].nsp` e `update.nsp`; `prod.keys` está na pasta compartilhada. **Não existem DLCs.** O NXNative lê a versão **real do update pelo content meta**, em vez de assumir `v0`, faz o perfil provisório (quando necessário), extração + merge base/update, rewrite ARM64 e shader split NVN onde aplicável.

**Download em chunks:** Colab e Actions usam `blades/drive_chunks.py`: 6 workers paralelos × 64 MiB, checagem de `Content-Range`, repetição individual da faixa e validação SHA-256/MD5. Se o Drive ignorar ranges, o fallback baixa em chunks sequenciais de 32 MiB. O upload é resumable em chunks de 64 MiB.

**iOS 26.0:** o Actions da branch `BladesTime` compila os shaders Metal reais e o framework, usa o Shared Shell e gera uma **IPA fina**: sem NSP, chaves ou RomFS dentro da IPA. A RomFS fica externa no Drive. O perfil provisório e o build **não garantem que o jogo abrirá**: será necessário testar no iPhone.

**Ordem:** execute as células **2, 4, 6, 8, 10 e 12** (contagem começando em 1, como no Dante). A célula 6 tenta recuperar o checkpoint. A célula 8 prepara o base+update somente se ainda não foi recuperado. A célula 12 publica **um handoff → um run**, exclusivamente na `BladesTime`.

**A célula 1 (transferência GitHub → Drive) só é executada no final**, depois que o workflow BladesTime tiver publicado uma IPA. Seleciona o workflow e a IPA do port correto, confere a proveniência e o hash do artifact e envia para **`botime/ipa`**.


In [ ]:
# O artefato é descoberto pela API do GitHub; nenhum ARTIFACT_ID/SHA fica fixado no notebook.

# BLADESTIME — TRANSFERÊNCIA AUTOMÁTICA GITHUB -> GOOGLE DRIVE
# Baixa o artefato diretamente no servidor Colab e envia a IPA ao Drive.
# Não recompila e não baixa os arquivos do jogo.

!pip -q install requests google-api-python-client google-auth

import os
import sys
import time
import hashlib
import zipfile
import requests
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# --------------------------------------------------
# CONFIGURAÇÃO — DESCOBRIR AUTOMATICAMENTE A ÚLTIMA IPA VÁLIDA
# --------------------------------------------------

REPO = "joao9vegeto-stack/KintTany"
BRANCH = "BladesTime"
SOURCE_ROOT_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
DESTINATION_FOLDER_ID = None  # resolved after authentication; points to botime
IPA_NAME = "BladesTime-BoTime.ipa"

WORK = Path("/content/bladestime_drive_transfer")
WORK.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORK / "github-artifact.zip"
EXTRACT_DIR = WORK / "extracted"

# --------------------------------------------------
# 1. AUTENTICAR GOOGLE DRIVE
# --------------------------------------------------

print("1/5 — Autenticando Google Drive...")
auth.authenticate_user()

credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)

drive = build(
    "drive", "v3",
    credentials=credentials,
    cache_discovery=False
)


# Resolve botime instead of using the old BOTW destination. Keep outputs isolated.
root_children = drive.files().list(
    q=f"'{SOURCE_ROOT_ID}' in parents and trashed = false",
    fields="nextPageToken,files(id,name,mimeType)", pageSize=1000
).execute().get("files", [])
botime_folders = [x for x in root_children if x.get("name", "").lower() == "botime"
                and x.get("mimeType") == "application/vnd.google-apps.folder"]
assert len(botime_folders) == 1, f"Expected exactly one botime subfolder: {botime_folders}"
DESTINATION_FOLDER_ID = botime_folders[0]["id"]
folder = drive.files().get(
    fileId=DESTINATION_FOLDER_ID,
    fields="id,name,mimeType"
).execute()

assert folder["mimeType"] == "application/vnd.google-apps.folder"
print("Drive autenticado:", folder["name"])

# --------------------------------------------------
# 2. DESCOBRIR A ÚLTIMA EXECUÇÃO BEM-SUCEDIDA DA BRANCH
# --------------------------------------------------

print(f"\n2/5 — Identificando a última IPA BoTime publicada na branch {BRANCH}...")

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "BladesTime-Colab-Transfer"
}
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

session = requests.Session()
session.headers.update(headers)
api_root = f"https://api.github.com/repos/{REPO}"

def github_get(url, params=None):
    r = session.get(url, params=params, timeout=(30, 90))
    if r.status_code in (401, 403):
        r.close()
        token = getpass("GitHub exige autenticação. Cole seu PAT (não será exibido): ").strip()
        if not token:
            raise RuntimeError("Token GitHub não fornecido.")
        session.headers["Authorization"] = f"Bearer {token}"
        r = session.get(url, params=params, timeout=(30, 90))
    r.raise_for_status()
    return r.json()

# A API lista execuções da branch; selecionamos especificamente o workflow BoTime.
# Assim, uma IPA de outro jogo/workflow não pode ser escolhida por engano.
runs = []
page = 1
while page <= 10:
    payload = github_get(
        f"{api_root}/actions/runs",
        params={"branch": BRANCH, "status": "success", "per_page": 100, "page": page}
    )
    batch = payload.get("workflow_runs", [])
    runs.extend(batch)
    if len(batch) < 100:
        break
    page += 1

# HEAD remoto atual da branch — não confundir com o commit da última IPA publicada.
latest_branch_commit = github_get(
    f"{api_root}/commits",
    params={"sha": BRANCH, "per_page": 1}
)[0]
LATEST_BRANCH_SHA = latest_branch_commit["sha"]
LATEST_BRANCH_MESSAGE = latest_branch_commit.get("commit", {}).get("message", "(mensagem indisponível)").splitlines()[0]
LATEST_BRANCH_DATE = latest_branch_commit.get("commit", {}).get("committer", {}).get("date", "(data indisponível)")

# Considera somente execuções concluídas com sucesso, na branch pedida,
# cujo workflow tenha BoTime no nome/caminho. Mais recentes primeiro.
runs = [
    r for r in runs
    if r.get("head_branch") == BRANCH
    and r.get("conclusion") == "success"
    and str(r.get("path", "")).split("@", 1)[0] == ".github/workflows/bladestime.yml"
]
runs.sort(key=lambda r: (r.get("created_at", ""), r.get("id", 0)), reverse=True)

selected_run = None
selected_artifact = None
for run in runs:
    artifacts = []
    artifacts_page = 1
    while artifacts_page <= 5:
        artifacts_payload = github_get(
            run["artifacts_url"],
            params={"per_page": 100, "page": artifacts_page}
        )
        batch = artifacts_payload.get("artifacts", [])
        artifacts.extend(batch)
        if len(batch) < 100:
            break
        artifacts_page += 1

    # Nome deve identificar BoTime; apenas nome genérico não é aceito.
    blades_artifacts = [
        a for a in artifacts
        if a.get("name", "").startswith("BladesTime-BoTime-") and not a.get("expired", False)
    ]
    if blades_artifacts:
        blades_artifacts.sort(key=lambda a: (a.get("created_at", ""), a.get("id", 0)), reverse=True)
        selected_run, selected_artifact = run, blades_artifacts[0]
        break

if not selected_run:
    raise RuntimeError(
        f"Nenhuma execução bem-sucedida do workflow BoTime com artefato BoTime não expirado "
        f"foi encontrada na branch {BRANCH}. Nenhuma IPA foi baixada."
    )

RUN_ID = selected_run["id"]
ARTIFACT_ID = selected_artifact["id"]
EXPECTED_ARTIFACT_SIZE = int(selected_artifact.get("size_in_bytes") or 0)
EXPECTED_ARTIFACT_SHA256 = str(selected_artifact.get("digest") or "").split(":", 1)[-1].lower()
COMMIT_SHA = selected_run["head_sha"]
COMMIT_SHORT = COMMIT_SHA[:12]
commit_details = github_get(f"{api_root}/commits/{COMMIT_SHA}")
COMMIT_MESSAGE = commit_details.get("commit", {}).get("message", "(mensagem indisponível)").splitlines()[0]
COMMIT_DATE = commit_details.get("commit", {}).get("committer", {}).get("date", "(data indisponível)")
RUN_URL = selected_run.get("html_url", f"https://github.com/{REPO}/actions/runs/{RUN_ID}")
COMMIT_URL = f"https://github.com/{REPO}/commit/{COMMIT_SHA}"
DRIVE_PROVENANCE_DESCRIPTION = (
    f"BladesTime BoTime provenance | branch={BRANCH} | commit={COMMIT_SHA} | "
    f"run_id={RUN_ID} | run_created_at={selected_run.get('created_at')} | "
    f"artifact={selected_artifact.get('name')} | artifact_id={ARTIFACT_ID} | "
    f"artifact_sha256={EXPECTED_ARTIFACT_SHA256} | "
    f"commit_url={COMMIT_URL} | run_url={RUN_URL}"
)

print("\n" + "=" * 68)
print("PROVENIÊNCIA DA IPA — IDENTIFICADA ANTES DO DOWNLOAD")
print("=" * 68)
print("Repositório:", REPO)
print("Branch consultada:", BRANCH)
print("HEAD ATUAL da branch:", LATEST_BRANCH_SHA)
print("Mensagem do HEAD:", LATEST_BRANCH_MESSAGE)
print("Data do HEAD:", LATEST_BRANCH_DATE)
print("\nCOMMIT QUE GEROU A IPA:")
print("SHA COMPLETO:", COMMIT_SHA)
print("SHA CURTO:", COMMIT_SHORT)
print("Mensagem:", COMMIT_MESSAGE)
print("Data do commit:", COMMIT_DATE)
print("Workflow:", selected_run.get("name"), "— run", RUN_ID)
print("Execução:", selected_run.get("created_at"), "→", selected_run.get("updated_at"))
print("Status:", selected_run.get("status"), "| conclusão:", selected_run.get("conclusion"))
print("URL da execução:", RUN_URL)
print("Artefato BoTime:", selected_artifact.get("name"), "— ID", ARTIFACT_ID)
print("Tamanho esperado do ZIP:", f"{EXPECTED_ARTIFACT_SIZE:,}", "bytes")
print("SHA-256 oficial do artefato:", EXPECTED_ARTIFACT_SHA256 or "não fornecido pela API")
print("Criado:", selected_artifact.get("created_at"))
print("Expira em:", selected_artifact.get("expires_at"))
print("URL do commit:", COMMIT_URL)
if COMMIT_SHA != LATEST_BRANCH_SHA:
    print("\nATENÇÃO: o commit que gerou a última IPA disponível NÃO é o HEAD atual.")
    print("A IPA selecionada é a publicação BoTime bem-sucedida mais recente com artefato válido.")
    print("Não é possível afirmar que ela inclui alterações posteriores ainda não compiladas.")
else:
    print("\nCONFIRMADO: a última IPA BoTime selecionada foi gerada pelo HEAD atual da branch.")
print("=" * 68 + "\n")

# Baixa exatamente o artefato associado à execução/commit exibidos acima.
# A listagem de workflows/artefatos pode ser pública, mas o endpoint de
# download exige autenticação com permissão de leitura de Actions.
api_url = f"{api_root}/actions/artifacts/{ARTIFACT_ID}/zip"
response = session.get(api_url, stream=True, timeout=(30, 180))

if response.status_code in (401, 403):
    response.close()
    print("\nO GitHub localizou o artefato, mas exige autenticação para baixar o ZIP.")
    print("Use um Personal Access Token com acesso ao repositório e permissão Actions: Read.")
    token = getpass("GitHub PAT (entrada oculta): ").strip()
    if not token:
        raise RuntimeError("Download cancelado: token GitHub não fornecido.")
    session.headers["Authorization"] = f"Bearer {token}"
    response = session.get(api_url, stream=True, timeout=(30, 180))

if response.status_code == 401:
    response.close()
    raise RuntimeError(
        "GitHub recusou o token (401). Gere/ajuste um PAT com permissão Actions: Read "
        "no repositório joao9vegeto-stack/KintTany e execute novamente."
    )
if response.status_code == 403:
    detail = response.text[:500]
    response.close()
    raise RuntimeError(
        "GitHub bloqueou o download (403). Verifique Actions: Read, acesso ao repositório "
        f"e limites da API. Resposta: {detail}"
    )
response.raise_for_status()
total = int(response.headers.get("Content-Length", 0))
downloaded = 0
last_report = -1
sha256 = hashlib.sha256()

with open(ZIP_PATH, "wb") as f:
    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
        if not chunk:
            continue
        f.write(chunk)
        sha256.update(chunk)
        downloaded += len(chunk)
        percent = int(downloaded * 100 / total) if total else 0
        if percent >= last_report + 5 or (total and percent == 100):
            total_text = f"/{total:,}" if total else ""
            print(f"GitHub -> Colab: {percent}% ({downloaded:,}{total_text} bytes)")
            last_report = percent
response.close()

assert ZIP_PATH.stat().st_size > 1_000_000, "Download muito pequeno; artefato inesperado."
DOWNLOADED_ZIP_SIZE = ZIP_PATH.stat().st_size
DOWNLOADED_ZIP_SHA256 = sha256.hexdigest().lower()

# Trava de proveniência: não extrai nem envia se tamanho/hash não forem
# exatamente os publicados pelo GitHub para o artefato selecionado.
if EXPECTED_ARTIFACT_SIZE and DOWNLOADED_ZIP_SIZE != EXPECTED_ARTIFACT_SIZE:
    raise RuntimeError(
        "INTEGRIDADE REPROVADA: o tamanho do ZIP baixado não corresponde ao GitHub. "
        f"Esperado={EXPECTED_ARTIFACT_SIZE:,}; recebido={DOWNLOADED_ZIP_SIZE:,}. "
        "A IPA NÃO será extraída nem enviada ao Drive."
    )
if EXPECTED_ARTIFACT_SHA256 and DOWNLOADED_ZIP_SHA256 != EXPECTED_ARTIFACT_SHA256:
    raise RuntimeError(
        "INTEGRIDADE REPROVADA: SHA-256 do ZIP diferente do digest oficial do GitHub. "
        f"Esperado={EXPECTED_ARTIFACT_SHA256}; recebido={DOWNLOADED_ZIP_SHA256}. "
        "A IPA NÃO será extraída nem enviada ao Drive."
    )

print("DOWNLOAD E INTEGRIDADE CONFIRMADOS.")
print("Artefato ID:", ARTIFACT_ID)
print("Commit confirmado:", COMMIT_SHA)
print("Tamanho ZIP confirmado:", f"{DOWNLOADED_ZIP_SIZE:,}", "bytes")
print("GitHub SHA-256:", EXPECTED_ARTIFACT_SHA256 or "não informado")
print("Download SHA-256:", DOWNLOADED_ZIP_SHA256)

# --------------------------------------------------
# 3. EXTRAIR SOMENTE A IPA
# --------------------------------------------------

print("\n3/5 — Extraindo IPA do artefato...")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Artifact ZIP inválido: CRC falhou em {bad_member}")
    ipa_members = [
        name for name in archive.namelist()
        if name.lower().endswith(".ipa")
        and not name.endswith("/")
    ]

    if len(ipa_members) != 1:
        raise RuntimeError(
            f"Esperada exatamente uma IPA; encontradas: {ipa_members}"
        )

    member = ipa_members[0]
    if Path(member).name != IPA_NAME:
        raise RuntimeError(f"IPA no artifact é de outro port: {member!r}, esperado {IPA_NAME!r}")
    ipa_path = EXTRACT_DIR / IPA_NAME

    with archive.open(member) as source, open(ipa_path, "wb") as target:
        while True:
            chunk = source.read(8 * 1024 * 1024)
            if not chunk:
                break
            target.write(chunk)

assert ipa_path.is_file()
ipa_size = ipa_path.stat().st_size
assert ipa_size > 1_000_000, "IPA extraída tem tamanho inválido."

print("IPA extraída:", ipa_path)
print("Tamanho:", f"{ipa_size:,}", "bytes")

# --------------------------------------------------
# 4. ENVIAR IPA DIRETAMENTE AO GOOGLE DRIVE
# --------------------------------------------------

print("\n4/5 — Preparando destino no Google Drive...")

query = (
    f"'{DESTINATION_FOLDER_ID}' in parents and "
    "name = 'ipa' and "
    "mimeType = 'application/vnd.google-apps.folder' "
    "and trashed = false"
)

found = drive.files().list(
    q=query, fields="files(id,name)", pageSize=100
).execute().get("files", [])

if found:
    ipa_folder_id = found[0]["id"]
else:
    created = drive.files().create(
        body={
            "name": "ipa",
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [DESTINATION_FOLDER_ID]
        },
        fields="id"
    ).execute()
    ipa_folder_id = created["id"]

# MD5 e SHA-256 locais para validar a integridade após o upload
md5 = hashlib.md5()
ipa_sha256_hash = hashlib.sha256()

with open(ipa_path, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        md5.update(chunk)
        ipa_sha256_hash.update(chunk)

local_md5 = md5.hexdigest()
IPA_SHA256 = ipa_sha256_hash.hexdigest()

# Atualiza uma IPA existente ou cria uma nova
query = (
    f"'{ipa_folder_id}' in parents and "
    f"name = '{IPA_NAME}' and trashed = false"
)

existing = drive.files().list(
    q=query, fields="files(id,name)", pageSize=100
).execute().get("files", [])

media = MediaFileUpload(
    str(ipa_path),
    mimetype="application/octet-stream",
    resumable=True,
    chunksize=64 * 1024 * 1024
)

if existing:
    request = drive.files().update(
        fileId=existing[0]["id"],
        body={"description": DRIVE_PROVENANCE_DESCRIPTION},
        media_body=media,
        fields="id,name,size,md5Checksum,description,webViewLink"
    )
    print("Atualizando IPA existente...")
else:
    request = drive.files().create(
        body={
            "name": IPA_NAME,
            "parents": [ipa_folder_id],
            "description": DRIVE_PROVENANCE_DESCRIPTION
        },
        media_body=media,
        fields="id,name,size,md5Checksum,description,webViewLink"
    )
    print("Criando IPA no Drive...")

uploaded = None
last_percent = -1

while uploaded is None:
    status, uploaded = request.next_chunk()
    if status:
        percent = int(status.progress() * 100)
        if percent >= last_percent + 5 or percent == 100:
            print(f"Colab -> Drive: {percent}%")
            last_percent = percent

# --------------------------------------------------
# 5. VERIFICAR UPLOAD NO DRIVE
# --------------------------------------------------

print("\n5/5 — Verificando arquivo no Google Drive...")

remote = drive.files().get(
    fileId=uploaded["id"],
    fields="id,name,size,md5Checksum,description,webViewLink"
).execute()

assert int(remote["size"]) == ipa_size, "Tamanho remoto diferente!"
assert remote["md5Checksum"] == local_md5, "MD5 remoto diferente!"
assert remote.get("description") == DRIVE_PROVENANCE_DESCRIPTION, "A proveniência não foi gravada no arquivo do Drive!"

print("\n======================================")
print("UPLOAD_VERIFICADO — SUCESSO")
print("======================================")
print("Arquivo:", remote["name"])
print("REPOSITÓRIO:", REPO)
print("BRANCH:", BRANCH)
print("COMMIT DA IPA (SHA COMPLETO):", COMMIT_SHA)
print("COMMIT DA IPA (SHA CURTO):", COMMIT_SHORT)
print("MENSAGEM DO COMMIT:", COMMIT_MESSAGE)
print("DATA DO COMMIT:", COMMIT_DATE)
print("HEAD ATUAL DA BRANCH:", LATEST_BRANCH_SHA)
print("COMMIT DA IPA É O HEAD?", "SIM" if COMMIT_SHA == LATEST_BRANCH_SHA else "NÃO — veja o aviso de proveniência acima")
print("WORKFLOW:", selected_run.get("name"))
print("WORKFLOW RUN:", RUN_ID)
print("RUN URL:", RUN_URL)
print("ARTIFACT:", selected_artifact.get("name"))
print("ARTIFACT ID:", ARTIFACT_ID)
print("ARTIFACT SHA-256 OFICIAL:", EXPECTED_ARTIFACT_SHA256 or "não informado")
print("ZIP SHA-256 BAIXADO:", DOWNLOADED_ZIP_SHA256)
print("Tamanho:", f'{int(remote["size"]):,}', "bytes")
print("IPA SHA-256:", IPA_SHA256)
print("MD5:", remote["md5Checksum"])
print("PROVENIÊNCIA GRAVADA NO DRIVE:", remote.get("description"))
print("Link direto:", remote.get("webViewLink"))
print(
    "Pasta BoTime:",
    "https://drive.google.com/drive/folders/" + DESTINATION_FOLDER_ID
)
print("======================================")

# Limpar ZIP e cópia temporária após verificação
ZIP_PATH.unlink(missing_ok=True)
ipa_path.unlink(missing_ok=True)

print("Temporários removidos do Colab.")


In [ ]:
import os, sys, argparse, json
from pathlib import Path
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/bladestime")
auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api_client = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
folder = drive_api_client.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType").execute()
assert folder.get("mimeType") == "application/vnd.google-apps.folder", folder
print("Drive autenticado:", folder.get("name"))
print("O BoTime será procurado exclusivamente na subpasta botime; nenhum jogo baixado nesta célula.")


## 1. Obter os scripts exclusivos do BladesTime

Clona somente a branch `BladesTime`. O clone de scripts não baixa os NSPs. O engine `nxnative` pinado e os patches iOS 26 da base Dante2iOS são preservados.


In [ ]:
%cd /content
!rm -rf /content/BladesTime-repo
!git clone --depth 1 --branch BladesTime https://github.com/joao9vegeto-stack/KintTany.git /content/BladesTime-repo
!git -C /content/BladesTime-repo rev-parse HEAD


## 2. Recuperar uma preparação BladesTime já concluída

Procura exclusivamente `botime/BladesTime_BoTime_*/handoff.json` no Drive. Recupera o handoff verificado sem baixar ou extrair base/update novamente. Se não houver, a etapa seguinte executa a preparação.


In [ ]:
%cd /content/BladesTime-repo
import importlib, hashlib, subprocess
from pathlib import Path
sys.path.insert(0, "/content/BladesTime-repo/blades/colab")
for name in ("source_archive", "prepare_linux", "run_colab"):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
repo_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="/content/BladesTime-repo", text=True).strip()
import source_archive
import run_colab
expected_source = Path("/content/BladesTime-repo/blades/colab/source_archive.py").resolve()
assert Path(source_archive.__file__).resolve() == expected_source
assert run_colab.BRANCH == "BladesTime" and run_colab.TITLE_ID == "0100CFA00CC74000"
assert '"botime"' in expected_source.read_text(encoding="utf-8")
print("Código BoTime:", repo_head)
print("SHA-256 source_archive.py:", hashlib.sha256(expected_source.read_bytes()).hexdigest())
RECOVERED_HANDOFF = run_colab.recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID), api=drive_api_client)
HANDOFF_READY = RECOVERED_HANDOFF is not None


## 3. Preparar a base + update no Colab (atualiza scripts automaticamente)

Esta célula busca o HEAD atualizado da `BladesTime`, recarrega os scripts e prepara **exatamente** o base+update da `botime`. Mantém o cache de `/content/bladestime` ao retomar. Se o engine materializado estiver incompleto, o backend aplica `engine.py sync --force`, sem apagar NSPs.

**Linux/Colab:** como no Dante original, o `nx_extract` de origem é Mach-O para macOS; o Colab compila o extrator C# original como **NativeAOT linux-x64** com .NET 10, valida `--source-sha`, armazena `/content/bladestime/native-tools/` e não altera a versão macOS do engine. O update precisa ter sua versão de content meta descoberta pelo extrator.


In [ ]:
# Sempre atualizar e recarregar o código antes da preparação.
# Isso evita o kernel do Colab executar a versão antiga dos scripts,
# mesmo que a branch BladesTime já esteja corrigida no GitHub.
import importlib, subprocess, sys
from pathlib import Path

BoTime_CHECKOUT = Path("/content/BladesTime-repo")
assert (BoTime_CHECKOUT / ".git").exists(), "Execute a célula 5 (clonar BladesTime) primeiro."
subprocess.run(["git", "fetch", "--depth", "1", "origin", "BladesTime"],
               cwd=BoTime_CHECKOUT, check=True)
latest_commit = subprocess.check_output(
    ["git", "rev-parse", "FETCH_HEAD"], cwd=BoTime_CHECKOUT, text=True).strip()
local_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=BoTime_CHECKOUT, text=True).strip()
if latest_commit != local_commit:
    print("[BladesTime] Atualizando scripts da branch:", local_commit[:12], "->", latest_commit[:12])
    subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"],
                   cwd=BoTime_CHECKOUT, check=True)
else:
    print("[BladesTime] Scripts já estão no HEAD:", latest_commit[:12])

source_folder = str(BoTime_CHECKOUT / "blades" / "colab")
if source_folder in sys.path:
    sys.path.remove(source_folder)
sys.path.insert(0, source_folder)
for name in ("run_colab", "prepare_linux", "source_archive"):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import run_colab
assert Path(run_colab.__file__).resolve() == Path(source_folder, "run_colab.py").resolve()
assert 'engine_command("sync", "--force")' in Path(run_colab.__file__).read_text(), (
    "Fonte antiga: atualize a branch e execute novamente esta célula.")
prep_linux_src = Path(source_folder, "prepare_linux.py").read_text(encoding="utf-8")
assert 'ensure_linux_native_extractor' in prep_linux_src and 'module.NX_EXTRACT = str(linux_nx_extract)' in prep_linux_src, (
    "Esta versão ainda usa o nx_extract macOS no Colab. Atualize a branch BladesTime.")
assert latest_commit == subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=BoTime_CHECKOUT, text=True
).strip(), "O clone local não está no HEAD obtido."
print("[BladesTime] Linux NativeAOT nx_extract habilitado, sem alterar o binário macOS.")
print("[BladesTime] Backend carregado:", run_colab.__file__)

if HANDOFF_READY:
    print("Preparação BoTime recuperada; nenhuma extração/rewrite foi refeita.")
else:
    RECOVERED_HANDOFF = run_colab.prepare(
        argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID), api=drive_api_client
    )
    HANDOFF_READY = (
        RECOVERED_HANDOFF.get("project") == "BladesTime"
        and RECOVERED_HANDOFF.get("title_id") == "0100CFA00CC74000"
        and RECOVERED_HANDOFF.get("package_version") == RECOVERED_HANDOFF.get("prepared", {}).get("package_version")
        and str(RECOVERED_HANDOFF.get("package_version", "")).startswith("v")
        and RECOVERED_HANDOFF.get("prepared", {}).get("input_package_count") == 2
        and RECOVERED_HANDOFF.get("prepared", {}).get("input_update_count") == 1
        and RECOVERED_HANDOFF.get("game_data", {}).get("aoc") == []
    )
    assert HANDOFF_READY, "Handoff BoTime incorreto ou incompleto"


## 4. Conferir checkpoint antes de disparar o Actions

Confere identidade do Title ID `0100CFA00CC74000`, a versão extraída **do update real** (não fixa em v0), dois NSPs (base + update), nenhum DLC, backend gráfico, hashes e links da RomFS/workspace externos. Se falhar, não despache.


In [ ]:
HANDOFF_PATH = Path("/content/bladestime-handoff.json")
assert HANDOFF_READY and HANDOFF_PATH.is_file(), "Preparação BoTime ainda não concluída"
HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
assert HANDOFF["project"] == HANDOFF["branch"] == "BladesTime"
assert HANDOFF["title_id"] == "0100CFA00CC74000"
import re
assert re.fullmatch(r"v[1-9][0-9]*", HANDOFF["package_version"]), "A versão do update NSP deve ser positiva"
assert HANDOFF["package_version"] == HANDOFF["prepared"]["package_version"]
assert HANDOFF["prepared"]["base_content_version"] == 0
assert HANDOFF["prepared"]["update_content_version"] == int(HANDOFF["package_version"][1:])
assert HANDOFF["engine_pin"] == "b526a50b85e4cfa3142c111e1ad491d9fd11f0eb"
assert HANDOFF["prepared"]["graphics"] in ("nvn", "gl", "vulkan")
assert HANDOFF["game_data"]["aoc"] == []
assert HANDOFF["prepared"]["input_package_count"] == 2
assert HANDOFF["prepared"]["input_update_count"] == 1
assert HANDOFF["prepared"]["input_dlc_count"] == 0
assert HANDOFF["game_data"]["romfs"]["id"] and HANDOFF["work_archive"]["id"]
print("BoTime base v0 + update validado. Backend:", HANDOFF["prepared"]["graphics"])
print("Perfil provisório:", HANDOFF["prepared"]["provisional"])
print("RomFS no Drive:", HANDOFF["game_data"]["folder_url"])
print("Handoff:", HANDOFF["drive_output_folder_url"])
print("Commit original do checkout:", HANDOFF["source_commit"])


## 5. Despachar um handoff → um workflow BladesTime

Use fine-grained GitHub PAT **Contents: Read and write**, somente para `KintTany`. **Esta célula publica apenas `blades/handoff.json` na branch `BladesTime`**. O workflow usa o secret já configurado `GOOGLE_SERVICE_ACCOUNT_JSON`, compila os shaders Metal e a IPA fina para iOS 26, e disponibiliza o artifact.

Depois que o Actions **terminar com sucesso**, execute a **célula 1 no topo** para transferir a IPA e registrar a proveniência em `botime/ipa`. Não execute a transferência antes de existir IPA.


In [ ]:
%cd /content/BladesTime-repo
!python blades/colab/run_colab.py dispatch --handoff /content/bladestime-handoff.json
